In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [3]:
df = pd.read_csv('../data/Bengaluru_House_Data.csv')

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (13320, 9)


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Shape: (13320, 9)

Columns:
['area_type', 'availability', 'location', 'size', 'society', 'total_sqft', 'bath', 'balcony', 'price']

Data types:
area_type        object
availability     object
location         object
size             object
society          object
total_sqft       object
bath            float64
balcony         float64
price           float64
dtype: object

Missing values:
area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64


In [5]:
print(df['price'].describe())

count    13320.000000
mean       112.565627
std        148.971674
min          8.000000
25%         50.000000
50%         72.000000
75%        120.000000
max       3600.000000
Name: price, dtype: float64


In [6]:
price_bins = [0, 50, 100, 200, 500, np.inf]
price_labels = ['Low', 'Medium', 'High', 'Very High', 'Luxury']

price_categories = pd.cut(
    df['price'],
    bins=price_bins,
    labels=price_labels,
    right=False
)

print(price_categories.value_counts().sort_index())

price
Low          3279
Medium       5761
High         2703
Very High    1313
Luxury        264
Name: count, dtype: int64


In [8]:
df['price_category'] = pd.cut(
    df['price'],
    bins=[0, 50, 100, 200, 500, np.inf],
    labels=['Low', 'Medium', 'High', 'Very High', 'Luxury'],
    right=False
)

print(df['price_category'].value_counts().sort_index())

price_category
Low          3279
Medium       5761
High         2703
Very High    1313
Luxury        264
Name: count, dtype: int64


In [9]:
X = df.drop(columns=['price', 'price_category'])
y = df['price_category']

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Features:
['area_type', 'availability', 'location', 'size', 'society', 'total_sqft', 'bath', 'balcony']

Target:
price_category


In [10]:
X = X.copy()

# Convert numeric-looking columns
X['total_sqft'] = pd.to_numeric(X['total_sqft'], errors='coerce')
X['size'] = X['size'].str.extract(r'(\d+)').astype(float)

# Fill missing numeric values
X['size'] = X['size'].fillna(X['size'].median())
X['bath'] = X['bath'].fillna(X['bath'].median())
X['balcony'] = X['balcony'].fillna(X['balcony'].median())
X['total_sqft'] = X['total_sqft'].fillna(X['total_sqft'].median())

# Fill missing categorical values
X['location'] = X['location'].fillna(X['location'].mode()[0])
X['society'] = X['society'].fillna('Unknown')

# One-hot encode categorical columns
X = pd.get_dummies(
    X,
    columns=['area_type', 'availability', 'location', 'society'],
    drop_first=True
)

print("Processed feature shape:", X.shape)
print("Missing values remaining:", X.isnull().sum().sum())

Processed feature shape: (13320, 4079)
Missing values remaining: 0


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

Training set: (10656, 4079)
Testing set: (2664, 4079)

Training target distribution:
price_category
Medium       4609
Low          2623
High         2162
Very High    1051
Luxury        211
Name: count, dtype: int64
